# Stage 22A: disjoint candidate-disagreement residual field

Stage 21のcandidate routerは終了します。この実験では候補を選ばず、A130 baseに対するA100/A160・selector・public OOFの行ごとの不一致とtarget-free軌跡特徴から非線形residualを学習します。Stage 21Aの63 wellsだけで学習し、Stage 21Bの58 wellsだけで固定評価します。well重複はゼロです。Kaggle提出は作りません。CPUランタイムを使用してください。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json,os,shutil,subprocess
REPOSITORY_URL='https://github.com/Okada-N13/rogii.git'
repo_dir=Path('/content/ROGII'); drive_root=Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir=drive_root/'artifacts'; data_dir=drive_root/'data'
if not (repo_dir/'.git').is_dir(): subprocess.run(['git','clone',REPOSITORY_URL,str(repo_dir)],check=True)
else: subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only','origin','main'],check=True)
if shutil.which('uv') is None: subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
os.environ['PATH']='/root/.local/bin:'+os.environ['PATH']
subprocess.run(['uv','sync','--frozen'],cwd=repo_dir,check=True)
assert (data_dir/'train').is_dir(),data_dir
def run_checked(command):
    result=subprocess.run(command,cwd=repo_dir,text=True,capture_output=True)
    if result.stdout: print(result.stdout,flush=True)
    if result.returncode:
        print(result.stderr,flush=True); raise RuntimeError(f'command failed: {command}')


## 固定された学習・検証split

Stage 21Aをtraining、Stage 21Bをvalidationとして再利用します。両者はwell単位で完全非重複です。outer suffix targetは学習ラベルまたは評価にだけ使い、特徴生成時のhidden-target invarianceも検査します。


In [ ]:
stage16b_run=artifact_dir/'stage16b_testlike_validation_full_v003'
stage17a_run=artifact_dir/'stage17_public_replay_full_v002'
public_oof_run=artifact_dir/'stage7_public_residual_gate_full_v001'
stage21a_run=artifact_dir/'stage21a_prefix_router_full_v001'
stage21b_run=artifact_dir/'stage21b_prefix_confidence_full_v001'
required=[stage16b_run/'well_assignments.parquet',stage17a_run/'cut_report.parquet',public_oof_run/'base_oof.parquet',stage21a_run/'router_cut_report.parquet',stage21b_run/'confidence_cut_report.parquet']
for path in required: assert path.is_file(),path
print(*required,sep='\n')


## Residual field

Stage 21よりPF計算は少なく、各cutでouter候補だけを生成します。学習特徴と検証をそれぞれ10 cutsごとに表示します。途中失敗時はこのrunだけを削除して最初から再実行します。


In [ ]:
RUN_ID='stage22a_residual_field_full_v001'; run_dir=artifact_dir/RUN_ID
if run_dir.exists() and not (run_dir/'summary.json').is_file():
    resolved=run_dir.resolve(); expected=(artifact_dir/RUN_ID).resolve()
    assert resolved==expected and resolved.parent==artifact_dir.resolve(),resolved
    print('Removing incomplete prior run:',resolved); shutil.rmtree(resolved)
if not (run_dir/'summary.json').is_file():
    run_checked(['uv','run','rogii-prefix-residual-field','--config','configs/experiment/stage22a_residual_field.yaml','--stage16b-run',str(stage16b_run),'--stage17a-run',str(stage17a_run),'--public-oof-run',str(public_oof_run),'--training-run',str(stage21a_run),'--validation-run',str(stage21b_run),'--data-dir',str(data_dir),'--artifact-dir',str(artifact_dir),'--run-id',RUN_ID])
summary=json.loads((run_dir/'summary.json').read_text())
{key:summary[key] for key in ['stage22a_complete','promoted_to_stage22b','training_cuts','training_wells','training_rows','validation_cuts','validation_wells','training_validation_well_overlap','feature_count','base_rmse','candidate_rmse','rmse_delta','well_p90_delta','bootstrap_95pct','gates','next_step']}


In [ ]:
import pandas as pd
pd.DataFrame(summary['weight_report']).sort_values('weight')


最後のsummary辞書とweight表を共有してください。primary weight 0.25の全gate通過時だけStage 22Bへ進みます。diagnostic weightを見てprimary判定を後付け変更しません。
